# Gene annotation and cross-dataset DEG integration

This notebook reproduces the annotation, filtering, integration, and top-200 prioritization used for the EBV-positive versus EBV-negative intestinal-type gastric cancer analysis. It uses dataset-specific GEO2R tables from GSE51575, GSE62254, and GSE66229.

Important methodological note: the retained historical analysis did **not** split multi-symbol annotations (for example, `GENE1 /// GENE2`) and did **not** collapse repeated gene symbols. It removed rows lacking a usable gene symbol, then ranked probe-level records by absolute log2 fold change. This behavior is reproduced faithfully below.

## Required input files

Place these six files either beside this notebook, in a `data` subfolder, or in the supplied `upload` folder:

- `GSE51575.top.table_Intestinal.tsv`
- `GSE62254_IIntestinal_EBV+vs_.top.table.tsv`
- `GSE66229.top.table.tsv`
- `All(1).xlsx` (retained annotation reference for GSE51575)
- `All_filtered(1).xlsx` (historical result used only for validation)
- `Top200_DEGs(1).xlsx` (historical result used only for validation)

In [ ]:
from pathlib import Path
import platform
import numpy as np
import pandas as pd

print('Python:', platform.python_version())
print('pandas:', pd.__version__)
print('numpy:', np.__version__)

In [ ]:
NOTEBOOK_DIR = Path.cwd()
SEARCH_DIRS = [NOTEBOOK_DIR, NOTEBOOK_DIR / 'data', NOTEBOOK_DIR / 'upload']
OUTPUT_DIR = NOTEBOOK_DIR / 'outputs_gene_annotation'
OUTPUT_DIR.mkdir(exist_ok=True)

def locate(filename):
    for folder in SEARCH_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'{filename} was not found in: {SEARCH_DIRS}')

FILES = {
    'GSE51575': locate('GSE51575.top.table_Intestinal.tsv'),
    'GSE62254': locate('GSE62254_IIntestinal_EBV+vs_.top.table.tsv'),
    'GSE66229': locate('GSE66229.top.table.tsv'),
    'legacy_all': locate('All(1).xlsx'),
    'legacy_filtered': locate('All_filtered(1).xlsx'),
    'legacy_top200': locate('Top200_DEGs(1).xlsx'),
}
for label, path in FILES.items():
    print(f'{label}: {path}')

## 1. Load and audit the three GEO2R tables

In [ ]:
raw = {name: pd.read_csv(FILES[name], sep='\t') for name in ['GSE51575', 'GSE62254', 'GSE66229']}
required = {'ID', 'adj.P.Val', 'P.Value', 'logFC'}
audit_rows = []
for dataset, table in raw.items():
    missing = required.difference(table.columns)
    if missing:
        raise ValueError(f'{dataset} is missing required columns: {sorted(missing)}')
    audit_rows.append({
        'dataset': dataset,
        'total_rows': len(table),
        'significant_rows_adj_P_Val_lt_0.05': int((table['adj.P.Val'] < 0.05).sum()),
        'gene_symbol_column_present': 'Gene.symbol' in table.columns,
    })
audit = pd.DataFrame(audit_rows)
audit

## 2. Apply the prespecified significance threshold

Rows with Benjamini-Hochberg adjusted p-value below 0.05 are retained. No additional fold-change threshold is imposed.

In [ ]:
significant = {name: table.loc[table['adj.P.Val'] < 0.05].copy() for name, table in raw.items()}
{name: len(table) for name, table in significant.items()}

## 3. Recover the retained probe-to-symbol annotation

The downloaded GSE62254 and GSE66229 GEO2R tables already contain `Gene.symbol`. The retained GSE51575 table contains sequence/accession fields but no gene-symbol column. To reproduce the original analysis without silently changing annotations, the three GSE51575 symbols are recovered from the preserved `All(1).xlsx` table by matching `ID`, `adj.P.Val`, and `logFC`. The preserved table is therefore an explicit supplementary input and provenance record for those three annotations.

In [ ]:
legacy_all = pd.read_excel(FILES['legacy_all'], sheet_name='DEG adj p-value <0.05')
legacy_all = legacy_all[['ID', 'adj.P.Val', 'logFC', 'symbol']].copy()

def standardize_with_existing_symbol(table, dataset):
    out = table[['ID', 'adj.P.Val', 'logFC', 'Gene.symbol']].copy()
    out = out.rename(columns={'Gene.symbol': 'symbol'})
    out.insert(0, 'dataset', dataset)
    return out

gse51575 = significant['GSE51575'][['ID', 'adj.P.Val', 'logFC']].merge(
    legacy_all, on=['ID', 'adj.P.Val', 'logFC'], how='left', validate='one_to_one'
)
gse51575.insert(0, 'dataset', 'GSE51575')
if gse51575['symbol'].isna().any():
    raise ValueError('At least one significant GSE51575 probe could not be matched to the retained annotation.')

annotated = {
    'GSE51575': gse51575,
    'GSE62254': standardize_with_existing_symbol(significant['GSE62254'], 'GSE62254'),
    'GSE66229': standardize_with_existing_symbol(significant['GSE66229'], 'GSE66229'),
}
annotated['GSE51575']

## 4. Integrate dataset-specific significant records

In [ ]:
integrated = pd.concat([annotated['GSE51575'], annotated['GSE62254'], annotated['GSE66229']], ignore_index=True)
integrated['symbol'] = integrated['symbol'].astype('string').str.strip()
integrated.loc[integrated['symbol'].eq(''), 'symbol'] = pd.NA
integrated['abs_logFC'] = integrated['logFC'].abs()
integrated['direction'] = np.where(integrated['logFC'] > 0, 'Up', np.where(integrated['logFC'] < 0, 'Down', 'No change'))
print('Integrated significant probe-level records:', len(integrated))
print('Records without a usable symbol:', int(integrated['symbol'].isna().sum()))
integrated.head()

## 5. Remove records without a usable gene symbol

Multi-symbol annotations are retained exactly as supplied by GEO2R. Repeated symbols/probes are also retained because that is what generated the historical 7,576-record table.

In [ ]:
filtered = integrated.loc[integrated['symbol'].notna()].copy().reset_index(drop=True)
summary = pd.DataFrame({
    'measure': [
        'Significant records before symbol filtering',
        'Records removed for missing/blank symbol',
        'Retained annotated probe-level records',
        'Upregulated records',
        'Downregulated records',
        'Distinct literal symbol labels',
        'Repeated symbol records beyond first occurrence',
        'Multi-symbol labels containing ///',
    ],
    'value': [
        len(integrated),
        int(integrated['symbol'].isna().sum()),
        len(filtered),
        int((filtered['logFC'] > 0).sum()),
        int((filtered['logFC'] < 0).sum()),
        filtered['symbol'].nunique(),
        int(filtered['symbol'].duplicated().sum()),
        int(filtered['symbol'].str.contains('///', regex=False, na=False).sum()),
    ]
})
summary

## 6. Prioritize the top 200 records by absolute log2 fold change

In [ ]:
top200 = filtered.sort_values('abs_logFC', ascending=False, kind='stable').head(200).reset_index(drop=True)
top200[['dataset', 'ID', 'adj.P.Val', 'logFC', 'symbol', 'abs_logFC', 'direction']].head(10)

## 7. Validate against the preserved historical outputs

Tied absolute fold changes may appear in a different order across sorting implementations; therefore, top-200 membership is validated as a set of records. The historical Excel file automatically converted nine gene symbols (`MARCH5`, `MARCH6`, `MARCH8`, `SEPT6`, `SEPT7`, `SEPT10`, and `SEPT11`) into dates. The present pipeline preserves the correct symbols from the original GEO2R text tables. Full-table record identity is therefore validated using probe ID and numerical statistics, while the date-conversion discrepancy is reported explicitly.

In [ ]:
legacy_filtered = pd.read_excel(FILES['legacy_filtered'], sheet_name='DEGs')
legacy_top200 = pd.read_excel(FILES['legacy_top200'], sheet_name='Sheet1')

def record_set(table, include_symbol=True):
    columns = [
        table['ID'].astype(str),
        table['adj.P.Val'].astype(float).round(12),
        table['logFC'].astype(float).round(12),
    ]
    if include_symbol:
        columns.append(table['symbol'].astype(str))
    return set(zip(*columns))

legacy_symbol_text = legacy_filtered['symbol'].astype(str)
excel_date_conversions = int(legacy_symbol_text.str.match(r'^2025-').sum())

checks = {
    'integrated_record_count_is_8372': len(integrated) == 8372,
    'filtered_record_count_is_7576': len(filtered) == 7576,
    'upregulated_record_count_is_3118': int((filtered['logFC'] > 0).sum()) == 3118,
    'downregulated_record_count_is_4458': int((filtered['logFC'] < 0).sum()) == 4458,
    'filtered_probe_and_statistics_match_historical_file': record_set(filtered, include_symbol=False) == record_set(legacy_filtered, include_symbol=False),
    'nine_historical_excel_symbol_to_date_conversions_detected': excel_date_conversions == 9,
    'top200_membership_matches_historical_file': record_set(top200) == record_set(legacy_top200),
}
validation = pd.Series(checks, name='passed').rename_axis('check').reset_index()
validation

In [ ]:
if not validation['passed'].all():
    failed = validation.loc[~validation['passed'], 'check'].tolist()
    raise AssertionError(f'Reproduction check(s) failed: {failed}')
print('All reproduction checks passed.')

## 8. Export machine-readable supplementary outputs

In [ ]:
audit.to_csv(OUTPUT_DIR / '01_input_audit.csv', index=False)
integrated.to_csv(OUTPUT_DIR / '02_all_significant_probe_records.tsv', sep='\t', index=False)
filtered.to_csv(OUTPUT_DIR / '03_annotated_probe_records.tsv', sep='\t', index=False)
top200.to_csv(OUTPUT_DIR / '04_top200_by_absolute_logFC.tsv', sep='\t', index=False)
summary.to_csv(OUTPUT_DIR / '05_processing_summary.csv', index=False)
validation.to_csv(OUTPUT_DIR / '06_reproduction_checks.csv', index=False)
print(f'Outputs saved to: {OUTPUT_DIR.resolve()}')
for path in sorted(OUTPUT_DIR.iterdir()):
    print('-', path.name)

## Reproducibility conclusion

The notebook reproduces 8,372 significant probe-level records before annotation filtering, removes 796 records without usable symbols, retains 7,576 annotated probe-level records (3,118 upregulated and 4,458 downregulated), and reproduces membership of the historical top-200 list.